# Assignment 2 - End-to-End ML Pipeline

- **Course:** MBAI 5310G: AI Programming - Ontario Tech University
- **Dataset:** E-commerce Customer Churn Dataset
- **Target:** 'Churned' (Yes / No) - Binary Classification

This notebook builds a complete ML pipeline to predict customer churn, from raw data to evaluated baseline model.

## Step 1: Load the Dataset

Load the CSV file into a pandas DataFrame.

In [1]:
import pandas as pd

df = pd.read_csv("ecommerce_customer_churn_dataset.csv")
df.head()

,Customer_ID,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type,Churned
0,1,37.0,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile,Yes
1,2,41.0,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop,No
2,3,30.0,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet,No
3,4,58.0,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile,Yes
4,5,59.0,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile,No


## Step 2: Inspect the Dataset

Check shape, column names, data types, and overall structure.

In [2]:
df.head()

,Customer_ID,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type,Churned
0,1,37.0,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile,Yes
1,2,41.0,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop,No
2,3,30.0,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet,No
3,4,58.0,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile,Yes
4,5,59.0,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile,No


In [3]:
df.shape

(325, 14)

In [4]:
df.columns

Index(['Customer_ID', 'Age', 'Gender', 'Region', 'Membership_Type',
       'Monthly_Spending', 'Number_of_Orders', 'Days_Since_Last_Purchase',
       'Customer_Support_Calls', 'Average_Rating', 'Used_Coupon',
       'Newsletter_Subscribed', 'Device_Type', 'Churned'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_ID               325 non-null    int64  
 1   Age                       320 non-null    float64
 2   Gender                    320 non-null    str    
 3   Region                    320 non-null    str    
 4   Membership_Type           320 non-null    str    
 5   Monthly_Spending          320 non-null    float64
 6   Number_of_Orders          325 non-null    int64  
 7   Days_Since_Last_Purchase  325 non-null    int64  
 8   Customer_Support_Calls    325 non-null    int64  
 9   Average_Rating            320 non-null    float64
 10  Used_Coupon               325 non-null    str    
 11  Newsletter_Subscribed     325 non-null    str    
 12  Device_Type               325 non-null    str    
 13  Churned                   325 non-null    str    
dtypes: float64(3), int64(

In [6]:
df.dtypes

Customer_ID                   int64
Age                         float64
Gender                          str
Region                          str
Membership_Type                 str
Monthly_Spending            float64
Number_of_Orders              int64
Days_Since_Last_Purchase      int64
Customer_Support_Calls        int64
Average_Rating              float64
Used_Coupon                     str
Newsletter_Subscribed           str
Device_Type                     str
Churned                         str
dtype: object

## Step 3: Understand the Problem

- **Business problem:** Identify customers who are likely to churn so the company can take retention action.
- **ML task:** Binary classification.
- **Prediction goal:** Predict the value of `Churned` (Yes / No) from customer attributes and behaviour.

In [7]:
business_problem = "An e-commerce company wants to predict which customers are likely to churn so it can take action to retain them."
ml_problem = "This is a binary classification problem because we want to predict whether the customer will churn or not."
prediction_goal = "Predict whether a customer will churn: Yes or No."

print("Business Problem:")
print(business_problem)
print("\nMachine Learning Problem:")
print(ml_problem)
print("\nPrediction Goal:")
print(prediction_goal)

Business Problem:
An e-commerce company wants to predict which customers are likely to churn so it can take action to retain them.

Machine Learning Problem:
This is a binary classification problem because we want to predict whether the customer will churn or not.

Prediction Goal:
Predict whether a customer will churn: Yes or No.


## Step 4: Define Features and Target

`X` = all customer attributes except `Customer_ID` (identifier only).
`y` = `Churned`.

In [8]:
feature_columns = [
    "Age",
    "Gender",
    "Region",
    "Membership_Type",
    "Monthly_Spending",
    "Number_of_Orders",
    "Days_Since_Last_Purchase",
    "Customer_Support_Calls",
    "Average_Rating",
    "Used_Coupon",
    "Newsletter_Subscribed",
    "Device_Type"
]

X = df[feature_columns]
y = df["Churned"]

print("Features (X):")
display(X.head())

print("Target (y):")
display(y.head())

Features (X):


,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type
0,37.0,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile
1,41.0,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop
2,30.0,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet
3,58.0,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile
4,59.0,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile


Target (y):


0    Yes
1     No
2     No
3    Yes
4     No
Name: Churned, dtype: str

In [9]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (325, 12)
Shape of y: (325,)


## Step 5: Data Cleaning

Handle missing values (median for numerical, mode for categorical), drop duplicates, and fix data types.

In [10]:
df_clean = df.copy()

print("Missing values before cleaning:")
print(df_clean.isnull().sum())

print("\nDuplicate rows before cleaning:", df_clean.duplicated().sum())

Missing values before cleaning:
Customer_ID                 0
Age                         5
Gender                      5
Region                      5
Membership_Type             5
Monthly_Spending            5
Number_of_Orders            0
Days_Since_Last_Purchase    0
Customer_Support_Calls      0
Average_Rating              5
Used_Coupon                 0
Newsletter_Subscribed       0
Device_Type                 0
Churned                     0
dtype: int64

Duplicate rows before cleaning: 5


In [11]:
numerical_columns = [
    "Age", "Monthly_Spending", "Number_of_Orders",
    "Days_Since_Last_Purchase", "Customer_Support_Calls", "Average_Rating"
]

categorical_columns = [
    "Gender", "Region", "Membership_Type",
    "Used_Coupon", "Newsletter_Subscribed", "Device_Type"
]

# Fill numerical missing values with median
for col in numerical_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Fill categorical missing values with mode
for col in categorical_columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Drop duplicates
df_clean = df_clean.drop_duplicates()

df_clean.head()

,Customer_ID,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type,Churned
0,1,37.0,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile,Yes
1,2,41.0,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop,No
2,3,30.0,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet,No
3,4,58.0,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile,Yes
4,5,59.0,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile,No


In [12]:
# Fix data types
df_clean["Customer_ID"] = df_clean["Customer_ID"].astype(int)
df_clean["Age"] = df_clean["Age"].astype(int)
df_clean["Monthly_Spending"] = df_clean["Monthly_Spending"].astype(float)
df_clean["Average_Rating"] = df_clean["Average_Rating"].astype(float)
df_clean["Number_of_Orders"] = df_clean["Number_of_Orders"].astype(int)
df_clean["Days_Since_Last_Purchase"] = df_clean["Days_Since_Last_Purchase"].astype(int)
df_clean["Customer_Support_Calls"] = df_clean["Customer_Support_Calls"].astype(int)

for col in categorical_columns + ["Churned"]:
    df_clean[col] = df_clean[col].astype("category")

print("Data types after cleaning:")
print(df_clean.dtypes)

Data types after cleaning:
Customer_ID                    int64
Age                            int64
Gender                      category
Region                      category
Membership_Type             category
Monthly_Spending             float64
Number_of_Orders               int64
Days_Since_Last_Purchase       int64
Customer_Support_Calls         int64
Average_Rating               float64
Used_Coupon                 category
Newsletter_Subscribed       category
Device_Type                 category
Churned                     category
dtype: object


In [13]:
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

print("\nDuplicate rows after cleaning:", df_clean.duplicated().sum())
print("Shape after cleaning:", df_clean.shape)

df_clean.head()

Missing values after cleaning:
Customer_ID                 0
Age                         0
Gender                      0
Region                      0
Membership_Type             0
Monthly_Spending            0
Number_of_Orders            0
Days_Since_Last_Purchase    0
Customer_Support_Calls      0
Average_Rating              0
Used_Coupon                 0
Newsletter_Subscribed       0
Device_Type                 0
Churned                     0
dtype: int64

Duplicate rows after cleaning: 0
Shape after cleaning: (320, 14)


,Customer_ID,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type,Churned
0,1,37,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile,Yes
1,2,41,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop,No
2,3,30,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet,No
3,4,58,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile,Yes
4,5,59,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile,No


In [14]:
df = df_clean.copy()

## Step 6: Preprocessing

Set up a `ColumnTransformer` to scale numerical features with `StandardScaler` and encode categorical features with `OneHotEncoder`. The transformer will be fit on training data only (after the split) to avoid data leakage.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df[feature_columns]
y = df["Churned"].map({"No": 0, "Yes": 1})

numerical_features = [
    "Age", "Monthly_Spending", "Number_of_Orders",
    "Days_Since_Last_Purchase", "Customer_Support_Calls", "Average_Rating"
]

categorical_features = [
    "Gender", "Region", "Membership_Type",
    "Used_Coupon", "Newsletter_Subscribed", "Device_Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [16]:
print("Features before preprocessing:")
display(X.head())

print("Target after Yes/No → 1/0 conversion:")
display(y.head())

Features before preprocessing:


,Age,Gender,Region,Membership_Type,Monthly_Spending,Number_of_Orders,Days_Since_Last_Purchase,Customer_Support_Calls,Average_Rating,Used_Coupon,Newsletter_Subscribed,Device_Type
0,37,Female,Manitoba,Platinum,332.0,1,218,7,1.0,No,No,Mobile
1,41,Female,Manitoba,Silver,632.0,11,199,3,3.0,No,Yes,Desktop
2,30,Male,British Columbia,Platinum,972.0,16,258,2,4.0,No,No,Tablet
3,58,Female,Manitoba,Basic,752.0,10,197,8,5.0,No,No,Mobile
4,59,Male,Quebec,Silver,619.0,12,81,3,4.0,No,Yes,Mobile


Target after Yes/No → 1/0 conversion:


0    1
1    0
2    0
3    1
4    0
Name: Churned, dtype: category
Categories (2, int64): [0, 1]

## Step 7: Train/Test Split

80% training, 20% testing, stratified by the target to preserve class balance.

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (256, 12)
X_test shape: (64, 12)
y_train shape: (256,)
y_test shape: (64,)


In [18]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train after preprocessing:", X_train_processed.shape)
print("X_test after preprocessing:", X_test_processed.shape)

X_train after preprocessing: (256, 24)
X_test after preprocessing: (64, 24)


## Step 8: Baseline Model - Logistic Regression

Train a Logistic Regression model on the preprocessed training data and generate predictions for the test set. A Decision Tree is also trained for additional comparison.

In [19]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_processed, y_train)

y_pred = baseline_model.predict(X_test_processed)
print("Predicted values:")
print(y_pred)

Predicted values:
[1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 0 0 1 0 1 0 1 0 0 0 0 0 0 1 1
 0 0 0 0 1 0 0 1 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 1 0]


In [20]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train_processed, y_train)

y_pred_tree = decision_tree_model.predict(X_test_processed)
print("Decision Tree predicted values:")
print(y_pred_tree)

Decision Tree predicted values:
[1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 1 1
 0 0 0 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0]


## Step 9: Evaluate the Baseline Model

Compute accuracy, confusion matrix, precision, recall, and F1-score on the test set.

In [21]:
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.84375


In [22]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[43  4]
 [ 6 11]]


In [23]:
cm_table = pd.DataFrame(
    cm,
    index=["Actual No (Not Churned)", "Actual Yes (Churned)"],
    columns=["Predicted No", "Predicted Yes"]
)
cm_table

,Predicted No,Predicted Yes
Actual No (Not Churned),43,4
Actual Yes (Churned),6,11


In [24]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Precision: 0.7333333333333333
Recall: 0.6470588235294118
F1-score: 0.6875


In [25]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No (Not Churned)", "Yes (Churned)"], zero_division=0))

Classification Report:
                  precision    recall  f1-score   support

No (Not Churned)       0.88      0.91      0.90        47
   Yes (Churned)       0.73      0.65      0.69        17

        accuracy                           0.84        64
       macro avg       0.81      0.78      0.79        64
    weighted avg       0.84      0.84      0.84        64



In [26]:
print("Model Evaluation Summary")
print("------------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Model Evaluation Summary
------------------------
Accuracy: 0.84375
Precision: 0.7333333333333333
Recall: 0.6470588235294118
F1-score: 0.6875


## Step 10: Interpret Results

The baseline Logistic Regression model achieves ≈84% accuracy on the test set. Precision (≈0.73) is higher than recall (≈0.65), which means the model is more cautious about flagging churn - it misses some real churners. Given the class imbalance (≈73% No / ≈27% Yes), F1-score (≈0.69) is a more honest indicator of performance than accuracy.

**Business interpretation:** the model can support retention campaigns by flagging at-risk customers, but the relatively low recall means a meaningful share of churners will be missed. Some important behavioural signals (browsing, cart events, session time) are not in this dataset and would likely improve performance.

In [27]:
print("Baseline Model Results")
print("----------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Baseline Model Results
----------------------
Accuracy: 0.84375
Precision: 0.7333333333333333
Recall: 0.6470588235294118
F1-score: 0.6875


## Step 11: Responsible AI Reflection

Brief slice analysis to check whether predicted churn rates differ meaningfully across `Gender` and `Region`. Any large gap would be a signal to investigate fairness before using the model operationally.

Key considerations:
- **Privacy** - the dataset contains personal attributes that must be handled responsibly.
- **Bias** - demographic features can lead to unequal treatment if not monitored.
- **Human review** - predictions should support, not replace, retention decisions.
- **Cost of errors** - false positives waste retention budget; false negatives lose customers.

In [28]:
results = X_test.copy()
results["Actual_Churned"] = y_test.values
results["Predicted_Churned"] = y_pred

gender_summary = results.groupby("Gender", observed=True)["Predicted_Churned"].mean()
print("Average predicted churn rate by Gender:")
print(gender_summary)

Average predicted churn rate by Gender:
Gender
Female    0.266667
Male      0.205882
Name: Predicted_Churned, dtype: float64


In [29]:
region_summary = results.groupby("Region", observed=True)["Predicted_Churned"].mean()
print("Average predicted churn rate by Region:")
print(region_summary)

Average predicted churn rate by Region:
Region
Alberta             0.500000
British Columbia    0.230769
Manitoba            0.062500
Ontario             0.071429
Quebec              0.428571
Name: Predicted_Churned, dtype: float64


## Summary

| Metric | Score |
|---|---:|
| Accuracy | 0.844 |
| Precision | 0.733 |
| Recall | 0.647 |
| F1-score | 0.688 |

The baseline Logistic Regression pipeline is fully reproducible end-to-end. F1-score is the most informative metric here due to class imbalance. A more advanced model comparison (Logistic Regression vs. SVM) is performed in Assignment 3.

## Final Explanation

**What the model is trying to predict.** Whether an e-commerce customer will churn, that is, stop purchasing from the platform. The target variable 'Churned' takes the value 'Yes' (churned) or 'No' (still active), encoded as '1' and '0' for modelling.

**Dataset used.** The E-commerce Customer Churn Dataset ('ecommerce_customer_churn_dataset.csv'), containing 325 customer records across 14 columns. After dropping duplicates, the working set was 320 customers.

**Features and target.**
- **Target:** 'Churned'
- **Features (12):** 'Age', 'Gender', 'Region', 'Membership_Type', 'Monthly_Spending', 'Number_of_Orders', 'Days_Since_Last_Purchase', 'Customer_Support_Calls', 'Average_Rating', 'Used_Coupon', 'Newsletter_Subscribed', 'Device_Type'.
- 'Customer_ID' was excluded because it is only an identifier and has no predictive value.

**Result obtained.** The Logistic Regression baseline achieved **Accuracy ≈ 0.84**, **Precision ≈ 0.73**, **Recall ≈ 0.65**, and **F1-score ≈ 0.69** on the held-out test set. The model is noticeably better at correctly identifying loyal customers than at catching customers who actually churn.

**One limitation.** The dataset is small (≈320 rows after cleaning) and class-imbalanced (≈73% No, ≈27% Yes), which makes recall on the minority class harder to optimise. The dataset also lacks behavioural signals such as browsing history, cart abandonment, and session duration, i.e. features that would likely improve churn prediction in a real e-commerce setting.